# Data Cleaning of "IntradaySummaries_.STOXX50E_19980220_20200809.csv"

**Source:** Data provided by Tobias Sichert through Dropbox in mail

## STOXX50E

In [ ]:
import pandas as pd 
file_path = "/Users/tobiasbergdahlpersson/Documents/SSE/MSc Thesis/Raw Data from Tobias Sichert/IntradaySummaries_.STOXX50E_19980220_20200809.csv"
df = pd.read_csv(file_path) 

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.describe()

In [ ]:
df.info()

### Producing a Copy of Data Frame

In [ ]:
# Copy of data frame
df_copy = df.copy()
# Quick sanity checks
print(df_copy.shape)
print(df_copy[["Date-Time", "Last"]].dtypes)
print(df_copy[["Date-Time", "Last"]].head(3))

In [ ]:
# What columns?
df_copy.columns

In [ ]:
# Making "Date-Time" column in date time
df_copy["Date-Time"] = pd.to_datetime(df_copy["Date-Time"], utc=True, errors="coerce")
print(df_copy["Date-Time"].dtype)
print(df_copy["Date-Time"].isna().sum())
print(df_copy["Date-Time"].head(3))

In [ ]:
# Min Max Date and Time in Sample
print(df_copy["Date-Time"].min())
print(df_copy["Date-Time"].max())

### Convert to Central European Time (CET)

In [ ]:
# Creating "Date-Time-CET" column
df_copy["Date-Time-CET"] = df_copy["Date-Time"].dt.tz_convert("Europe/Berlin")
df_copy[["Date-Time", "Date-Time-CET"]].head(3)

### Diagnostic Checking

**Comment:** During which hours does the index actually have price activity?

In [ ]:
df_copy["Hour_CET"] = df_copy["Date-Time-CET"].dt.hour
df_copy.groupby("Hour_CET")["Last"].count()

**Comment:** Computing the fraction of minutes where "Last" changes within each hour

In [ ]:
df_copy["Last_change"] = df_copy["Last"].ne(df_copy["Last"].shift(1))
df_copy.groupby("Hour_CET")["Last_change"].mean()

In [ ]:
# How many non-missing prices per hour relative to total minutes?
hour_total = df_copy.groupby("Hour_CET")["Last"].size()
hour_nonmissing = df_copy.groupby("Hour_CET")["Last"].count()
(hour_nonmissing / hour_total)

**Why is it ~0.69 and not ~1.0?**

Because your dataset is a continuous minute grid across 22 years, including:

* Weekends

* Holidays

* Non-trading days

* Possibly overnight minutes

So when we divide by all calendar minutes, trading activity looks diluted.

This is expected.

#### Remove weekends

In [ ]:
df_copy["Weekday"] = df_copy["Date-Time-CET"].dt.weekday  # 0=Mon, 6=Sun
df_copy["Weekday"].value_counts().sort_index()

In [ ]:
# Remove weekends 
df_copy = df_copy[df_copy["Weekday"] < 5].copy() # 5 and 6 are Saturday and Sunday
df_copy.shape

In [ ]:
# Recompute activity intensity per hour: 
hour_total = df_copy.groupby("Hour_CET")["Last"].size()
hour_nonmissing = df_copy.groupby("Hour_CET")["Last"].count()
(hour_nonmissing / hour_total) 

In [ ]:
df_copy[df_copy["Hour_CET"] == 17]["Last"].isna().mean()

Therefore, trading window should be 09:00 - 17:30. 

In [ ]:
df_copy[df_copy["Hour_CET"] == 17]["Date-Time-CET"].dt.minute.value_counts().sort_index().head(20)

In [ ]:
df_copy_17 = df_copy[(df_copy["Hour_CET"] == 17) & (df_copy["Last"].notna())]
df_copy_17["Date-Time-CET"].dt.minute.value_counts().sort_index()

This implies that trading collapses after 17:30, suggesting trading window should strictly be 09:00 - 17:30. 
This matches European cash equity structure:

Continuous trading ends at 17:30 CET

#### Market Microstructure Diagnostics

**After 17:30:**

* Closing auction mechanics

* Post-close index updates

* Sporadic recalculations

* Vendor artifacts

The sharp drop after minute 30 is exactly what we expect if trading ends at 17:30.

**Why?**

* 09–16 → full continuous trading

* 17:00–17:29 → valid continuous trading

* 17:30 onward → non-continuous, sparse, structurally different

**Including post-17:30 minutes would:**

* Artificially distort realized volatility

* Introduce microstructure noise

* Mix auction mechanics with continuous trading

Most academic HF volatility papers restrict to continuous trading hours only.

### Retrieving the correct trading window (09:00 - 17:30)

This implies that we will have weekdays only, continuous trading hours only, no overnight trades, no post-close artifacts. 

In [ ]:
df_copy = df_copy[
    (df_copy["Hour_CET"] >= 9) &
    (
        (df_copy["Hour_CET"] < 17) |
        ((df_copy["Hour_CET"] == 17) & (df_copy["Date-Time-CET"].dt.minute < 30))
    )
].copy()

df_copy.shape

### Set the datetime index correctly

In [ ]:
df_copy = df_copy.sort_values("Date-Time-CET").set_index("Date-Time-CET")
type(df_copy.index), df_copy.index.dtype, df_copy.index[:3]

In [ ]:
# Create a daily "Date" key
df_copy["Date"] = df_copy.index.floor("D")
df_copy[["Date", "Last"]].head(3)

### Potentially removing or not removing holidays

In [ ]:
dec24 = df_copy[(df_copy.index.month == 12) & (df_copy.index.day == 24)]
dec24.groupby("Date")["Last"].count().describe()

In [ ]:
jan1 = df_copy[(df_copy.index.month == 1) & (df_copy.index.day == 1)]
jan1.groupby("Date")["Last"].count().describe()

# Final CSV for work

In [ ]:
df_final_output = df_copy[["Date", "Last", "Volume", "No. Trades"]].copy()
df_final_output.head()

In [ ]:
output_path = "/Users/tobiasbergdahlpersson/Documents/SSE/MSc Thesis/Cleaned Data Sets/EUROSTOXX50.csv"
#df_final_output.to_csv(output_path, index=True)

#### For momentum factor in MALL feature set 

In [ ]:
# In your first notebook — save daily close prices
daily_close = df_copy["Last_filled"].groupby(df_copy["Date"]).last()
daily_close.name = "Close"
daily_close.to_csv(
    "/Users/tobiasbergdahlpersson/Documents/SSE/MSc Thesis/Cleaned Data Sets/daily_close.csv",
    header=True
)
print(f"Saved {len(daily_close)} daily closing prices")
print(daily_close.head())